# OpenWeather PM2.5 comparisons

This notebook downloads air-pollution records for rice samples or district centroids and builds monthly PM2.5 comparisons.
The paper's main PM2.5 source is CAMS in [`replication/extract_covariates.py`](replication/extract_covariates.py).

## Setup

Start Jupyter from the repository root with `requests`, `pandas`, `geopandas`, `shapely`, and `tqdm`.
Provide `datasets/South_sample_5x10000.csv` with `sample_4`, `lat`, and `lon`.
The district branch needs `JAXA_subtiles.json` and `gadm41_VNM_2.json`; review these root-level paths against the current `shapefiles/` layout.
Create `datasets/owm/` and `owm_pollution/` before downloads.
Set `OPENWEATHER_API_KEY` in the kernel environment; no credential is stored here.

## Branches and execution

Sections 1 and 3 process the 2022 sample.
Section 2 downloads districts for 2018-2022 into a separate folder.
Download cells make external requests and overwrite JSONs.
`time.mktime` uses the local timezone, with endpoints at midnight on the last named day.
District centroids use geographic coordinates; downloading resumes at index 290.

Cleanup cleared debug outputs and execution counts without rerunning downloads or analysis.
See [`NOTEBOOKS.md`](NOTEBOOKS.md) for structural and syntax checks.


## 1. Sampled rice locations

### Load dependencies and API key

Only downloads need the key; existing JSONs can be aggregated offline.


In [ ]:
import os
import json
import time
from datetime import datetime
from pathlib import Path

import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from tqdm.notebook import tqdm

dir = os.getcwd()
api_key = os.environ.get("OPENWEATHER_API_KEY", "")

### Load the sample table

Uses the five-by-10,000 table; the three-percent alternative is commented.


In [ ]:
# df = pd.read_csv(os.path.join(dir, 'datasets/South_VNM_3per_sampleID.csv'))
df = pd.read_csv(os.path.join(dir, 'datasets/South_sample_5x10000.csv'))
df

### Select sample four

Filters `sample_4 == 1` without drawing a new sample.


In [ ]:
# Filter by sample
sample = df[df['sample_4'] == 1]
sample

### Download 2022 records

Requests January 1-December 31 using local Unix timestamps and writes coordinate JSONs to `datasets/owm/`.
Check API errors before aggregation.
The district section replaces this `create_json` definition.


In [ ]:
lats = sample['lat'].to_list()
lons = sample['lon'].to_list()

def create_json(lat:str, lon:str, start: int, end: int) -> dict:
    """Makes a request to the OpenWeatherMap API to get air pollution data for a given location and time period."""
    if not api_key:
        raise ValueError("Set OPENWEATHER_API_KEY before requesting data")
    hist_url = f'https://api.openweathermap.org/data/2.5/air_pollution/history?lat={lat}&lon={lon}&start={start}&end={end}&appid={api_key}'
    r = requests.get(hist_url)
    json_out = r.json()

    return json_out

# Loop through the coordinates and create a json file for each pair
for i in tqdm(range(len(lats))[:]):

    print(lats[i], lons[i])


    start_1 = int(time.mktime(datetime(2022, 1, 1).timetuple()))
    end_1 = int(time.mktime(datetime(2022, 12, 31).timetuple()))

    # Dumps json file
    json_1 = create_json(lats[i], lons[i], start_1, end_1)
    with open(f"datasets/owm/{lats[i]}_{lons[i]}_2022.json", "w") as outfile:
        json.dump(json_1, outfile)

## 2. District centroids

### Select tiles

Uses JSON positions 1-15 and 33 onward and writes WGS84 polygons to `chosen_tiles.json`.
These slices do not establish full study-area coverage.


In [ ]:
# JAXA LULC tiles
with open('JAXA_subtiles.json', 'r') as json_file:
    jaxa_subtiles = json.load(json_file)

tiles = list(jaxa_subtiles.keys())[1:16]
tiles.extend(list(jaxa_subtiles.keys())[33:])

# Bounding box of the chosen tiles
bbox = [jaxa_subtiles[tile] for tile in tiles]
bbox = [box(tile[0], tile[1], tile[2], tile[3]) for tile in bbox]

# Create a GeoDataFrame from the tiles and their bounding boxes
tiles_gdf = gpd.GeoDataFrame({'tile': tiles,
                              'geometry': bbox})
tiles_gdf = tiles_gdf.set_crs(epsg=4326)
tiles_gdf.head()
tiles_gdf.to_file('chosen_tiles.json', driver='GeoJSON', index=False)

### Load districts

Uses GADM 4.1 level 2 in the tiles' coordinate system.


In [ ]:
# Loads VNM's districts
dist_gdf = gpd.read_file('gadm41_VNM_2.json')
dist_gdf = dist_gdf.to_crs(tiles_gdf.crs)
dist_gdf.head()

### Select districts and centroids

Keeps districts intersecting any tile and computes centroids in EPSG:4326.
Set conversion and source order affect the resume slice.


In [ ]:
# Check which districts intersect with the chosen tiles
dist_geom = [geom for geom in dist_gdf.geometry]

intersects_geom = []
for geom in dist_geom:
    gdf = tiles_gdf.iloc[tiles_gdf.sindex.query(geom, predicate='intersects')]
    if len(gdf) > 0:
        intersects_geom.append(geom)

intersects_geom = list(set(intersects_geom))
dist_filtered = dist_gdf[dist_gdf.geometry.isin(intersects_geom)]

# Get the centroid of the districts
dist_filtered['lon'] = dist_filtered.centroid.x
dist_filtered['lat'] = dist_filtered.centroid.y
dist_filtered = dist_filtered[['GID_2', 'lat', 'lon']]
dist_dict = dist_filtered.set_index('GID_2').to_dict('index')

### Resume district downloads

Redefines `create_json` for district IDs and resumes at index 290.
Writes `HN_...` files for two windows per year, 2018-2022.
June 30 and December 31 endpoints are local midnight, excluding the rest of those days.


In [ ]:
def create_json(location_name:str, start: int, end: int) -> dict:
    """Makes a request to the OpenWeatherMap API to get air pollution data for a given district and time period."""
    global dist_dict
    lat = dist_dict[location_name]['lat']
    lon = dist_dict[location_name]['lon']
    if not api_key:
        raise ValueError("Set OPENWEATHER_API_KEY before requesting data")
    hist_url = f'https://api.openweathermap.org/data/2.5/air_pollution/history?lat={lat}&lon={lon}&start={start}&end={end}&appid={api_key}'
    r = requests.get(hist_url)
    json_out = r.json()
    json_out['name'] = location_name

    return json_out

for location_name in list(dist_dict.keys())[290:]:

    print(location_name)

    # 6 month extraction to avoid request timeout from OWM

    for year in list(range(2018, 2023, 1)):
        print('     ', year)
        # January - June
        start_1 = int(time.mktime(datetime(year, 1, 1).timetuple()))
        end_1 = int(time.mktime(datetime(year, 6, 30).timetuple()))

        # Dumps json file
        json_1 = create_json(location_name, start_1, end_1)
        with open(f"owm_pollution/HN_{location_name}_16_{year}.json", "w") as outfile:
            json.dump(json_1, outfile)

        # July - December
        start_2 = int(time.mktime(datetime(year, 7, 1).timetuple()))
        end_2 = int(time.mktime(datetime(year, 12, 31).timetuple()))

        # Dumps json file
        json_2 = create_json(location_name, start_2, end_2)
        with open(f"owm_pollution/HN_{location_name}_712_{year}.json", "w") as outfile:
            json.dump(json_2, outfile)

## 3. Aggregate sample JSONs

Reads `datasets/owm/latitude_longitude_2022.json`, parsing basenames independently of parent folders.
Converts UTC timestamps to `Asia/Saigon` and averages each file by month.
Grouping omits year: multiyear files combine same-numbered months, while separate years create duplicate location/month keys.
Joins the sampling table on exact numeric latitude/longitude.


In [ ]:
# All the json files in the directory
paths = sorted(Path('datasets/owm').glob('*.json'))
paths.sort()
dfs = []

# Loop through the json files and extract the data
# Creates a dataframe for each json file
# Concatenates the list into a single dataframe
for path in tqdm(paths):
    with open(path, 'r') as f:
        d = json.load(f)
    dates = []
    pm25s = []

    for i in range(len(d['list'])):
        dates.append(d['list'][i]['dt'])
        pm25s.append(d['list'][i]['components']['pm2_5'])


    latitude, longitude, year_label = path.stem.rsplit('_', 2)

    df_p = pd.DataFrame({'lat': latitude,
                        'lon': longitude,
                        'dt': dates,
                        'pm25': pm25s})
    df_p['dt'] = pd.to_datetime(df_p['dt'], unit='s')
    df_p['dt'] = df_p['dt'].dt.tz_localize('UTC').dt.tz_convert('Asia/Saigon')
    df_p['month'] = df_p['dt'].dt.month
    df_p = df_p.groupby(['lat', 'lon', 'month']).agg({'pm25': 'mean'}).reset_index()

    df_p['lat'] = df_p['lat'].astype('float64')
    df_p['lon'] = df_p['lon'].astype('float64')

    dfs.append(df_p)

df_all = pd.concat(dfs, axis=0).reset_index(drop=True)

# Merges the sample data with the pollution data
df_out = pd.merge(df, df_all, on=['lat', 'lon'], how='inner')

### Write the comparison table

Writes `datasets/owm_pm25_2022_2nd_batch.csv` despite selecting `sample_4`.


In [ ]:
# Saves the output to a csv file
df_out.to_csv('datasets/owm_pm25_2022_2nd_batch.csv', index=False)